In [58]:
from dotenv import load_dotenv
load_dotenv()

True

In [59]:
from sqlalchemy import (
    create_engine,
    Column,
    Integer,
    String,
    Float,
    ForeignKey,
    DateTime
)
from sqlalchemy.orm import relationship, sessionmaker, DeclarativeBase
from datetime import datetime

In [60]:
# Base Model
class Base(DeclarativeBase):
    pass

# 创建表格
class Customer(Base):
    __tablename__ = "customers"
    id = Column(Integer, primary_key=True)
    name = Column(String, nullable=False)

    orders = relationship("Order", back_populates="customer")


class FoodItem(Base):
    __tablename__ = "food_items"
    id = Column(Integer, primary_key=True)
    name = Column(String, nullable=False)
    price = Column(Float, nullable=False)

    orders = relationship("Order", back_populates="food_item")

class Order(Base):
    __tablename__ = "orders"
    id = Column(Integer, primary_key=True)
    customer_id = Column(Integer, ForeignKey("customers.id"), nullable=False)
    food_item_id = Column(Integer, ForeignKey("food_items.id"), nullable=False)
    order_date = Column(DateTime, default=datetime.utcnow)
    delivery_address = Column(String, nullable=False)

    customer = relationship("Customer", back_populates="orders")
    food_item = relationship("FoodItem", back_populates="orders")

engine = create_engine("sqlite:///mydatabase.db") # 此时数据库是空的
Base.metadata.create_all(engine) # 将数据的“图纸集”交给 engine，执行 CREATE_TABLE 语句


In [61]:
Session = sessionmaker(bind=engine) # 告诉 Session，应该连接到哪个数据库；sessionmaker是一个工厂函数，生产一个能创建连接的“类”。sessionmaker在配置需要绑定的数据库引擎，这一步只需要做一次。

In [62]:
from contextlib import contextmanager

@contextmanager
def get_session():
    session = Session()
    try:
        yield session
        session.commit()
    except Exception:
        session.rollback()
        raise
    finally:
        session.close()

In [63]:
# # 创建 customer 数据，运行一次
# with get_session() as session:
#     new_customer = Customer(name="John Doe")
#     session.add(new_customer)

# pizza1 = FoodItem(name="Pizza Margherita", price=8.5)
# pizza2 = FoodItem(name="Pizza Salami", price=9.5)
# pizza3 = FoodItem(name="Pizza Quattro Formaggi", price=10.5)

# with get_session() as session:
#     session.add_all([pizza1, pizza2, pizza3])

In [64]:
with get_session() as session:
    customer = session.query(Customer).first()
    print(customer.name)
    foods = session.query(FoodItem).all()
    for food in foods:
        print("{fname} priced at {price}".format(fname=food.name, price=food.price))

John Doe
Pizza Margherita priced at 8.5
Pizza Salami priced at 9.5
Pizza Quattro Formaggi priced at 10.5


In [65]:
from langchain_classic.schema import Document
from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import os

In [66]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/text-embedding-004",
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

In [67]:
docs = [
    Document(
        page_content="the dog loves to eat pizza", metadata={"source": "animal.txt"}
    ),
    Document(
        page_content="the cat loves to eat lasagna", metadata={"source": "animal.txt"}
    ),
]

db = Chroma.from_documents(
    docs,
    embeddings,
    persist_directory="./data/chroma_db",
)
retriever = db.as_retriever(search_kwargs={"k": 2})

def format_docs(docs: list[Document]):
    return "\n\n".join(doc.page_content for doc in docs)


In [68]:
template = """Answer the question based only on the following content
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)
llm = ChatOpenAI(
    model="deepseek-chat",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com",
    temperature=0
)

retrieval_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [69]:
retrieval_chain.invoke(input="What food does the cat like")

'Based only on the given content, the cat likes lasagna.'

In [70]:
from typing import TypedDict
from langchain_core.messages import BaseMessage

chain_with_prompt = prompt | llm | StrOutputParser()

In [71]:
class AgentState(TypedDict):
    question: str
    raw_docs: list[BaseMessage]
    formatted_docs: str
    generation: str

In [72]:
def get_docs(state: AgentState):
    print("get_docs: ", state)
    question = state["question"]
    docs = retriever.invoke(question)
    state["raw_docs"] = docs
    return state

In [73]:
def format_docs(state: AgentState):
    print("format docs: ", state)
    state["formatted_docs"] = "\n\n".join([doc.page_content for doc in state["raw_docs"]])
    return state

In [74]:
def generate(state: AgentState):
    question = state["question"]
    formatted_docs = state["formatted_docs"]
    result = chain_with_prompt.invoke({"question": question, "context": formatted_docs})
    state["generation"] = result
    return state

In [75]:
from langgraph.graph import StateGraph, END

workflow = StateGraph(AgentState)

workflow.add_node("get_docs", get_docs)
workflow.add_node("format_docs", format_docs)
workflow.add_node("generate", generate)

workflow.add_edge("get_docs", "format_docs")
workflow.add_edge("format_docs", "generate")
workflow.add_edge("generate", END)

workflow.set_entry_point("get_docs")

app = workflow.compile()


In [82]:
result = app.invoke({"question": "What food does the cat like?"})

get_docs:  {'question': 'What food does the cat like?'}
format docs:  {'question': 'What food does the cat like?', 'raw_docs': [Document(metadata={'source': 'animal.txt'}, page_content='the cat loves to eat lasagna'), Document(metadata={'source': 'animal.txt'}, page_content='the cat loves to eat lasagna')]}


In [83]:
result["generation"]

'Based on the content provided, the cat likes lasagna.'